<a href="https://colab.research.google.com/github/hamza-24-ai/Pytorch_with_practical_deepLearning/blob/main/06_ANN_Optuna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import torch.optim as optim
from torch.utils.data import Dataset,DataLoader

# Checking Data

In [2]:
df = pd.read_csv("/content/Colab_data/train.csv")
print("Checking data rows and columns")
print(df.shape)
print("\n print 1st five rows of data ")
print(df.head())

Checking data rows and columns
(103904, 25)

 print 1st five rows of data 
   Unnamed: 0      id  Gender      Customer Type  Age   Type of Travel  \
0           0   70172    Male     Loyal Customer   13  Personal Travel   
1           1    5047    Male  disloyal Customer   25  Business travel   
2           2  110028  Female     Loyal Customer   26  Business travel   
3           3   24026  Female     Loyal Customer   25  Business travel   
4           4  119299    Male     Loyal Customer   61  Business travel   

      Class  Flight Distance  Inflight wifi service  \
0  Eco Plus              460                      3   
1  Business              235                      3   
2  Business             1142                      2   
3  Business              562                      2   
4  Business              214                      3   

   Departure/Arrival time convenient  ...  Inflight entertainment  \
0                                  4  ...                       5   
1          

In [6]:
# Checking if there is null value
print("Checking null values ")
print(df.isnull().sum())
print("\n Checking data types")
print(df.dtypes)

Checking null values 
Unnamed: 0                             0
id                                     0
Gender                                 0
Customer Type                          0
Age                                    0
Type of Travel                         0
Class                                  0
Flight Distance                        0
Inflight wifi service                  0
Departure/Arrival time convenient      0
Ease of Online booking                 0
Gate location                          0
Food and drink                         0
Online boarding                        0
Seat comfort                           0
Inflight entertainment                 0
On-board service                       0
Leg room service                       0
Baggage handling                       0
Checkin service                        0
Inflight service                       0
Cleanliness                            0
Departure Delay in Minutes             0
Arrival Delay in Minutes           

In [8]:
# Arrival delay in minutes have empty rows fill this

df["Arrival Delay in Minutes"] = df["Arrival Delay in Minutes"].fillna(df["Arrival Delay in Minutes"].median())
print("Row are filled Successfully")
print(df.isnull().sum())

Row are filled Successfully
Unnamed: 0                           0
id                                   0
Gender                               0
Customer Type                        0
Age                                  0
Type of Travel                       0
Class                                0
Flight Distance                      0
Inflight wifi service                0
Departure/Arrival time convenient    0
Ease of Online booking               0
Gate location                        0
Food and drink                       0
Online boarding                      0
Seat comfort                         0
Inflight entertainment               0
On-board service                     0
Leg room service                     0
Baggage handling                     0
Checkin service                      0
Inflight service                     0
Cleanliness                          0
Departure Delay in Minutes           0
Arrival Delay in Minutes             0
satisfaction                        

In [10]:
# Drop Unnecessary Columns

df = df.drop(columns = ["id"])
print("Columns are dropped Successfully")
print(df.isnull().sum())


Columns are dropped Successfully
Unnamed: 0                           0
Gender                               0
Customer Type                        0
Age                                  0
Type of Travel                       0
Class                                0
Flight Distance                      0
Inflight wifi service                0
Departure/Arrival time convenient    0
Ease of Online booking               0
Gate location                        0
Food and drink                       0
Online boarding                      0
Seat comfort                         0
Inflight entertainment               0
On-board service                     0
Leg room service                     0
Baggage handling                     0
Checkin service                      0
Inflight service                     0
Cleanliness                          0
Departure Delay in Minutes           0
Arrival Delay in Minutes             0
satisfaction                         0
dtype: int64


In [11]:

categorical_cols = ['Gender', 'Customer Type', 'Type of Travel', 'Class','satisfaction']

numerical_cols = ['Age', 'Flight Distance', 'Departure Delay in Minutes', 'Arrival Delay in Minutes']

le = LabelEncoder()
scaler = StandardScaler()

for cat in categorical_cols:
  df[cat] = le.fit_transform(df[cat])

df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

print("Columns are Label Encoding and Standard Scaler Done")
print(df.head())


Columns are Label Encoding and Standard Scaler Done
   Unnamed: 0  Gender  Customer Type       Age  Type of Travel  Class  \
0           0       1              0 -1.745279               1      2   
1           1       1              1 -0.951360               0      0   
2           2       0              0 -0.885200               0      0   
3           3       0              0 -0.951360               0      0   
4           4       1              0  1.430397               0      0   

   Flight Distance  Inflight wifi service  Departure/Arrival time convenient  \
0        -0.731539                      3                                  4   
1        -0.957184                      3                                  2   
2        -0.047584                      2                                  2   
3        -0.629246                      2                                  5   
4        -0.978244                      3                                  3   

   Ease of Online booking  .

# Splitting Columns

In [23]:
# Split the cols in models

X = df.drop(columns = ["satisfaction"])
y = df["satisfaction"]

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

print("Models are splitting Successfully")
print(df.shape)

Models are splitting Successfully
(103904, 24)


In [13]:
# Checking if GPU is Running

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device} is runinng")

Device : cuda is runinng


In [30]:
# Create Custom Class data

class CustomData(Dataset):
  def __init__(self,features,labels):
    self.features = torch.tensor(features.values, dtype=torch.float32)
    self.labels = torch.tensor(labels.values, dtype=torch.long)

  def __len__(self):
    return len(self.features)

  def __getitem__(self,index):
    return self.features[index], self.labels[index]

In [31]:
# Create Dataset on Train and test

train_dataset = CustomData(X_train,y_train)
test_dataset = CustomData(X_test,y_test)


In [32]:
print("Train data length")
print(len(X_train))
print(f"Test Data Length \n {len(X_test)}")

Train data length
83123
Test Data Length 
 20781


In [33]:
# Create nn Forward data
class MyNN(nn.Module):

  def __init__(self,input_dim,output_dim,num_hidden,num_neurons,dropout):

    super().__init__()

    layers=[]

    for i in range(num_hidden):

      layers.append(nn.Linear(input_dim,num_neurons))
      layers.append(nn.BatchNorm1d(num_neurons))
      layers.append(nn.ReLU())
      layers.append(nn.Dropout(dropout))
      input_dim = num_neurons

    layers.append(nn.Linear(input_dim,output_dim))

    self.model = nn.Sequential(*layers)

  def forward(self,x):
    return self.model(x)

# Objective WorkFlow

In [40]:
def objective(trial):

  # Create plan of action
  num_hidden = trial.suggest_int("num_hidden", 2,6)
  num_neurons = trial.suggest_int("num_neurons", 64, 256, step=64)
  num_epochs = trial.suggest_int("num_epochs", 10, 60, step=10)
  optimizer_name = trial.suggest_categorical("optimizer", ["SGD", "Adam", "RMSProp"])
  learning_rate = trial.suggest_float("learning_rate", 1e-5 , 1e-1, log=True)
  batch = trial.suggest_categorical("batch", [64,128,256])
  dropout_rate = trial.suggest_float("dropout_rate", 0.1,0.5, step=0.1)
  weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)

  train_loader = DataLoader(train_dataset, batch_size=batch, shuffle=True, pin_memory=True)
  test_loader = DataLoader(test_dataset, batch_size=batch, shuffle=True, pin_memory=True)


  # model Init
  input_dim = 23
  output_dim = 2

  model = MyNN(input_dim,output_dim,num_hidden,num_neurons,dropout_rate)
  model.to(device)

  # Param init
  criterian = nn.CrossEntropyLoss()

  if optimizer_name == "Adam":
    optimizer = optim.Adam(model.parameters(), lr = learning_rate, weight_decay = weight_decay)
  elif optimizer_name == "SGD":
    optimizer = optim.SGD(model.parameters(), lr = learning_rate, weight_decay = weight_decay)
  else:
    optimizer = optim.RMSprop(model.parameters(), lr = learning_rate, weight_decay = weight_decay)

  # traning Loop

  for epoch in range(num_epochs):

    for batch_features, batch_labels in train_loader:
      #  Moving to gpu

      batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

      # Forward Pass

      outputs = model(batch_features)

      # loss
      loss = criterian(outputs, batch_labels)

      # back propagation

      optimizer.zero_grad()
      loss.backward()

      # update grads

      optimizer.step()


  # evaluation
  model.eval()

  # evaluation on test data
  total = 0
  correct = 0

  with torch.no_grad():

    for batch_features, batch_labels in test_loader:

      # move data to gpu
      batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

      outputs = model(batch_features)

      _, predicted = torch.max(outputs, 1)

      total = total + batch_labels.shape[0]

      correct = correct + (predicted == batch_labels).sum().item()

    accuracy = correct/total

  return accuracy

In [25]:
# install optuna

!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 20.1 MB/s eta 0:00:00


In [38]:
import optuna

study = optuna.create_study(direction='maximize')

[I 2026-08-22 20:33:22,383] A new study created in memory with name: no-name-52e76452-fec1-4998-a902-eaa1360d7954


In [42]:
study.optimize(objective, n_trials=10)

[I 2026-08-22 20:39:08,398] Trial 2 finished with value: 0.854434339059718 and parameters: {'num_hidden': 5, 'num_neurons': 64, 'num_epochs': 40, 'optimizer': 'RMSProp', 'learning_rate': 0.00014683759436527795, 'batch': 64, 'dropout_rate': 0.1, 'weight_decay': 3.020110280969574e-05}. Best is trial 2 with value: 0.854434339059718.
[I 2026-08-22 20:40:49,225] Trial 3 finished with value: 0.5514652807853327 and parameters: {'num_hidden': 6, 'num_neurons': 128, 'num_epochs': 20, 'optimizer': 'SGD', 'learning_rate': 8.004854061723445e-05, 'batch': 64, 'dropout_rate': 0.1, 'weight_decay': 0.0003587399149272151}. Best is trial 2 with value: 0.854434339059718.
[I 2026-08-22 20:41:43,990] Trial 4 finished with value: 0.8262836244646552 and parameters: {'num_hidden': 2, 'num_neurons': 256, 'num_epochs': 30, 'optimizer': 'RMSProp', 'learning_rate': 0.00028929429020764193, 'batch': 128, 'dropout_rate': 0.2, 'weight_decay': 5.114313156859258e-05}. Best is trial 2 with value: 0.854434339059718.
[I 2

In [43]:
study.best_value

0.855492998412011

In [44]:
study.best_params

{'num_hidden': 2,
 'num_neurons': 256,
 'num_epochs': 60,
 'optimizer': 'Adam',
 'learning_rate': 0.00028557987149707546,
 'batch': 64,
 'dropout_rate': 0.1,
 'weight_decay': 9.919043454052938e-05}